# 06 — Product demonstration

A concise, interactive view of data readiness, anomaly evidence, incident
consolidation and probable fault location. This notebook visualises saved
outputs; it does not refit or retune the model.


## 1. Load one completed run


In [ ]:
import os
import sys
from pathlib import Path

import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
DATA_ROOT = Path(
    os.getenv("ANOMALY_DATA_ROOT") or os.getenv("ANOMALY_DRIVE_ROOT")
    or ("/content/drive/MyDrive/anomaly_detection" if IN_COLAB
        else Path.home() / "anomaly_detection_data")
).expanduser()
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DATA_ROOT / "research" / "milestone1" if IN_COLAB
    else Path.cwd() if (Path.cwd() / "milestone1_core.py").is_file()
    else Path.cwd() / "notebooks" / "drive_research",
)).expanduser()
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))
from milestone1_core import CORE_VERSION, read_json

SECTOR = os.getenv("ANOMALY_SECTOR", "telecom")
CANONICAL_RUN_IDS = {
    "telecom": "telecom_core_v0_11_run1",
    "petrobras_3w": "petrobras_3w_core_v0_11_run1",
}
RUN_ROOT = DATA_ROOT / "outputs" / "canonical" / f"v{CORE_VERSION}" / SECTOR / os.getenv(
    "CANONICAL_RUN_ID", CANONICAL_RUN_IDS[SECTOR]
)
CORE_ROOT = RUN_ROOT / "SPEC-CORE"
VERSION = "3.0.0"
EDA_ROOT = DATA_ROOT / "outputs" / "eda" / f"v{VERSION}" / SECTOR / f"{SECTOR}_eda_v3_run1"
MODEL_ROOT = DATA_ROOT / "outputs" / "models" / f"v{VERSION}" / SECTOR / f"{SECTOR}_models_v3_run1"
CASE_ROOT = DATA_ROOT / "outputs" / "cases" / f"v{VERSION}" / SECTOR / f"{SECTOR}_cases_v3_run1"
EVALUATION_ROOT = DATA_ROOT / "outputs" / "evaluation" / f"v{VERSION}" / SECTOR / f"{SECTOR}_evaluation_v3_run1"

metric_summary = pd.read_csv(EDA_ROOT / "metric_summary.csv")
comparison = pd.read_csv(MODEL_ROOT / "development_comparison.csv")
configuration = read_json(MODEL_ROOT / "selected_configuration.json")
incidents = pd.read_csv(CASE_ROOT / "ranked_incidents.csv")
metrics = pd.read_csv(CASE_ROOT / "evaluation_metrics.csv")
fault_types = pd.read_csv(CASE_ROOT / "fault_type_results.csv")
domain_types = pd.read_csv(CASE_ROOT / "domain_type_results.csv")
localisation = pd.read_csv(CASE_ROOT / "localisation_results.csv")
trace = pd.read_parquet(CASE_ROOT / "incident_score_trace.parquet")
incident_run = read_json(CASE_ROOT / "incident_run.json")
resolution = pd.read_csv(EVALUATION_ROOT / "statistical_resolution.csv")
display(pd.Series({
    "sector": SECTOR, "partition": incident_run["partition"],
    "portfolio": configuration["portfolio"], "incidents": len(incidents),
    "topology_enabled": configuration["topology_enabled"],
}, name="value").to_frame())


## 2. Data quality and a representative series


In [ ]:
quality = metric_summary[["metric_id", "invalid_rate", "clipped_rate"]].melt(
    "metric_id", var_name="quality", value_name="fraction"
)
px.bar(
    quality, x="metric_id", y="fraction", color="quality", barmode="group",
    title="Calibration data quality by metric",
).show()

telemetry_glob = str(CORE_ROOT / "telemetry" / "part-*.parquet")
metric_id = metric_summary.sort_values("rows", ascending=False).iloc[0].metric_id
with duckdb.connect() as connection:
    representative = connection.execute('''
        SELECT CAST(entity_id AS VARCHAR), CAST(episode_id AS VARCHAR), count(*) AS rows
        FROM read_parquet(?)
        WHERE metric_id = ? AND quality_code <> 'invalid' AND value IS NOT NULL
        GROUP BY entity_id, episode_id ORDER BY rows DESC LIMIT 1
    ''', [telemetry_glob, metric_id]).fetchone()
    series = connection.execute('''
        SELECT event_ts, value FROM read_parquet(?)
        WHERE metric_id = ? AND entity_id = ? AND episode_id = ?
        ORDER BY event_ts LIMIT 20000
    ''', [telemetry_glob, metric_id, representative[0], representative[1]]).df()
px.line(series, x="event_ts", y="value",
        title=f"{metric_id} — representative {representative[0]}").show()


## 3. Development trade-off at a common workload


In [ ]:
figure = px.scatter(
    comparison, x="false_case_rate_ci_high", y="event_recall",
    color="portfolio", symbol="threshold_quantile", size="cases",
    hover_data=["case_precision", "joint_detection_localisation", "median_delay_seconds"],
    title="Development recall versus conservative false-incident workload",
)
figure.add_vline(x=configuration["budget_gate"], line_dash="dash")
figure.show()
display(resolution)
display(domain_types)
consolidation = metrics.loc[metrics.metric.isin([
    "raw_alert_count", "consolidated_case_count",
    "alert_volume_reduction", "mean_alerts_per_case",
])]
display(consolidation)
px.bar(
    fault_types.sort_values("recall"), x="recall", y="fault_type",
    orientation="h", color="reporting_status",
    hover_data=["detected_faults", "scoreable_faults", "joint_detection_localisation_recall"],
    title="Detection and localisation evidence by fault type",
).show()


## 4. Ranked incidents and evidence trace


In [ ]:
shown = incidents.head(20).copy()
px.bar(
    shown.sort_values("rank", ascending=False),
    x="anomaly_evidence_score", y="case_id", orientation="h",
    color="scope_type", hover_data=[
        "case_start", "channels", "leading_features",
        "affected_fraction_estimate", "identifiability_status",
    ], title="Highest statistical-evidence incidents",
).show()
display(shown[[
    "rank", "case_id", "case_start", "scope_type", "scope_id",
    "scope_type_2", "scope_id_2", "affected_entity_count",
    "affected_fraction_estimate", "identifiability_status",
    "anomaly_evidence_score", "channels", "location_explanation",
]])

if not shown.empty and not trace.empty:
    top_case = shown.iloc[0].case_id
    evidence = trace.loc[trace.case_id.eq(top_case)]
    figure = go.Figure()
    for channel, frame in evidence.groupby("channel"):
        figure.add_trace(go.Scatter(x=frame.event_ts, y=frame.anomaly_score, name=channel))
        figure.add_trace(go.Scatter(
            x=frame.event_ts, y=frame.threshold,
            name=f"{channel} threshold", line={"dash": "dash"},
        ))
    figure.update_layout(title=f"{top_case}: score versus frozen calibration threshold")
    figure.show()


    transformed = evidence.dropna(
        subset=["leading_feature", "observed_transformed", "expected_transformed"]
    )
    if not transformed.empty:
        focus = transformed.sort_values("anomaly_score", ascending=False).iloc[0]
        detail = transformed.loc[
            transformed.entity_id.eq(focus.entity_id)
            & transformed.leading_feature.eq(focus.leading_feature)
        ].sort_values("event_ts")
        figure = go.Figure()
        figure.add_trace(go.Scatter(
            x=detail.event_ts, y=detail.observed_transformed,
            name="observed transformed value",
        ))
        figure.add_trace(go.Scatter(
            x=detail.event_ts, y=detail.expected_transformed,
            name="frozen calibration reference", line={"dash": "dash"},
        ))
        figure.update_layout(
            title=f"{top_case}: {focus.entity_id} / {focus.leading_feature}"
        )
        figure.show()


## 5. Localisation scorecard and claim boundary


In [ ]:
localisation_metrics = metrics.loc[
    metrics.metric.str.contains("localisation|footprint|hierarchy", case=False, regex=True)
]
display(localisation_metrics)
display(localisation.loc[localisation.detected].head(20))
display(Markdown(f'''
- **Result status:** {incident_run['partition']} evidence.
- **Location meaning:** probable observable topology scope, not proven root cause.
- **Evidence limit:** {incident_run['synthetic_limit']}.
- **Operational priority:** not calculated until validated impact data is available.
- **Holdout discipline:** development selects the configuration; holdout is opened once.
'''))
